![Cabecera](../../assets/cabecera_rag.png)

# Workout 2 - Retrieval y búsqueda semántica

## Objetivos

En el Workout anterior **indexaste** fragmentos en ChromaDB. Ahora simulas lo que ocurre cuando un usuario **hace una pregunta**:

1. Convertir la pregunta en un vector (embedding).
2. Buscar en Chroma los chunks **más parecidos** (similarity search).
3. Quedarte con los **top-K** mejores y formatear el **contexto recuperado**.

> **Importante:** todavía **no** generamos respuesta con un LLM. Solo recuperamos texto del corpus. La generación es Sprint 10.

**Prerrequisito:** haber ejecutado el [Workout 1](../01_Bases_de_datos_vectoriales/01_crear_base_vectorial_chromadb.ipynb) para que exista `../output/chroma_db/`.

## Setup — Instalar librerías

Solo necesitamos tres paquetes en este workout:

- **chromadb** — leer el índice y hacer `collection.query()`.
- **google-genai** — embeddear la pregunta del usuario.

Dependencias a instalar:

In [ ]:
%pip install -q chromadb google-genai

## Variables y carga de datos de output de la etapa anterior

Apuntamos a la misma carpeta y colección que utilizamos anteriormente:

- `CHROMA_DIR` → `../output/chroma_db/`
- `COLLECTION_NAME` → `agenda_cultural_madrid`
- `TOP_K = 3` → devolveremos los 3 chunks más cercanos a la pregunta.

Si la carpeta no existe, la celda fallará con un mensaje claro: debes ejecutar el Workout 1 primero.

También configuramos **`GEMINI_API_KEY`**: el embed de la pregunta debe usar el **mismo modelo** que al indexar (`gemini-embedding-2`).

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types

WORKOUT_DIR = Path("..").resolve()
CHROMA_DIR = WORKOUT_DIR / "output" / "chroma_db"
COLLECTION_NAME = "agenda_cultural_madrid"
EMBEDDING_MODEL = "gemini-embedding-2"
TOP_K = 3

load_dotenv(WORKOUT_DIR / ".env")
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ")

assert CHROMA_DIR.exists(), f"No existe {CHROMA_DIR}. Ejecuta el Workout 1 primero."
print("ChromaDB:", CHROMA_DIR)

## Paso 1 — Embeddear la pregunta del usuario

El retrieval no compara texto con texto directamente: compara **vector con vector**.

Por eso la pregunta `"¿Hay cine gratuito en verano?"` pasa por la misma API de embeddings que usaste para los chunks.

La función `embeddear_consulta()` devuelve una lista de números (p. ej. 3072 valores). Ese es el **punto de búsqueda** en el espacio semántico.

In [2]:
def embeddear_consulta(client: genai.Client, pregunta: str) -> list[float]:
    """Convierte la pregunta del usuario en un vector (mismo modelo que al indexar)."""
    contents = [types.Content(parts=[types.Part(text=pregunta)])]
    result = client.models.embed_content(model=EMBEDDING_MODEL, contents=contents)
    return list(result.embeddings[0].values)


client = genai.Client()
pregunta = "¿Hay cine gratuito en verano?"

vector = embeddear_consulta(client, pregunta)

print(f"Pregunta: {pregunta}")
print(f"Dimensiones del vector: {len(vector)}")

Pregunta: ¿Hay cine gratuito en verano?
Dimensiones del vector: 3072


## Paso 2 — Similarity search (top-K)

Con el vector de la pregunta, Chroma busca los embeddings más cercanos en la colección.

- **`n_results=TOP_K`** — cuántos chunks devolver (3 en este ejemplo).
- **`distances`** — qué tan lejos está cada chunk (menor distancia → más similar con coseno).
- **`metadatas`** — de qué archivo vino cada fragmento (`source`).

Lee la salida pensando: *¿estos fragmentos ayudarían a responder la pregunta?* No busques aún una respuesta redactada.

In [3]:
# Reabrir el índice creado en el Workout 1
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma.get_collection(COLLECTION_NAME)
print("Documentos en índice:", collection.count())

# Cuántos resultados pedir (no más de los que existen en la colección)
num_resultados = min(TOP_K, collection.count())

# Búsqueda por similitud: comparamos el vector de la pregunta con los del índice
results = collection.query(
    query_embeddings=[vector],  # lista porque Chroma admite varias preguntas a la vez
    n_results=num_resultados,
    include=["documents", "metadatas", "distances"],
)

# Chroma devuelve listas anidadas: [0] = resultados de la PRIMERA (y única) pregunta
ids = results["ids"][0]
documents = results["documents"][0]
metadatas = results["metadatas"][0]
distances = results["distances"][0]

# Mostrar cada chunk recuperado, del más similar al menos similar
for i, doc_id in enumerate(ids):
    distancia = distances[i]
    fuente = Path(metadatas[i].get("source", "?")).name
    texto_preview = documents[i][:250]

    print(f"\n#{i + 1} {doc_id}  distance={distancia:.4f}")
    print("fuente:", fuente)
    print(texto_preview, "...")

Documentos en índice: 20

#1 chunk_13  distance=0.3536
fuente: 206974-4-agenda-eventos-culturales-100-csv.csv
Evento: 10 vidas
Descripción: Becket es un gato adoptado a quien los mimos de su ama, Rose, lo han vuelto egoísta y malcriado. Un día pierde su novena vida y, a las puertas del cielo, se niega a aceptar que su estancia en la Tierra termine. Por ello  ...

#2 chunk_0  distance=0.4195
fuente: 206974-3-agenda-eventos-culturales-100.pdf
Estructura para Eventos provenientes de www.madrid.es


La información existente en este conjunto de datos, proviene de la página web municipal www.madrid.es.
Esta estructura de información, es genérica para todos l os conjuntos de datos de eventos y ...

#3 chunk_19  distance=0.4285
fuente: 206974-4-agenda-eventos-culturales-100-csv.csv
Evento: 90 años de la declaración del Parque del Retiro como Bien de Interés Cultural
Actividad: 90 años de la declaración del Parque del Retiro como Bien de Interés Cultural
Lugar: Centro de Educación Ambiental El 

## Paso 3 — Construir el contexto recuperado

A continuación, **formateamos** el contexto recuperado para inspeccionarlos. En Sprint 10 pegarás estos fragmentos en un prompt para Gemini. 

Buenas prácticas que aplicamos:

- **Delimitadores** (`--- Fragmento N ---`) entre chunks.
- **Citar la fuente** (`faq_...`, `csv_...`) para trazabilidad.
- **Mostrar la distancia** en modo debug (útil para comparar calidad).

El resultado es un único string `contexto` listo para copiar o enviar a un LLM más adelante.

In [4]:
def formatear_contexto(results: dict) -> str:
    """Convierte la respuesta de Chroma en un bloque de texto legible."""
    # Extraemos listas de la primera (y única) consulta
    ids = results["ids"][0]
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    partes = []
    for i, doc_id in enumerate(ids):
        texto = documents[i]
        meta = metadatas[i]
        dist = distances[i]
        fuente = Path(meta.get("source", "?")).name

        bloque = (
            f"--- Fragmento {i + 1} (dist={dist:.4f}) ---\n"
            f"Fuente: {fuente}\n"
            f"{texto}"
        )
        partes.append(bloque)

    if not partes:
        return "(sin resultados)"
    return "\n\n".join(partes)


contexto = formatear_contexto(results)
print(contexto[:1200], "\n..." if len(contexto) > 1200 else "")

--- Fragmento 1 (dist=0.3536) ---
Fuente: 206974-4-agenda-eventos-culturales-100-csv.csv
Evento: 10 vidas
Descripción: Becket es un gato adoptado a quien los mimos de su ama, Rose, lo han vuelto egoísta y malcriado. Un día pierde su novena vida y, a las puertas del cielo, se niega a aceptar que su estancia en la Tierra termine. Por ello suplica por una segunda oportunidad y le conceden nueve vidas más. Lo que el gat o no sabe es que en cada una de ellas se reencarnará en un animal diferente. Duración: 88 minutos. Recomendada para mayores de 7 años.
Actividad: Cine de verano en Hortaleza
Lugar: Parque de Villa Rosa-Paco Caño
Distrito: HORTALEZA
Fecha: 2026-08-21 00:00:00.0
Hora: 22:00
Gratuito: sí

--- Fragmento 2 (dist=0.4195) ---
Fuente: 206974-3-agenda-eventos-culturales-100.pdf
Estructura para Eventos provenientes de www.madrid.es


La información existente en este conjunto de datos, proviene de la página web municipal www.madrid.es.
Esta estructura de información, es genérica para 

## Paso 4 — Probar otra pregunta

Un mismo índice responde a **muchas preguntas distintas**. Cambia la consulta y observa si cambian las fuentes recuperadas.

Esta pregunta sobre el campo **GRATUITO** debería recuperar **documentación del dataset** (FAQ o PDF de campos), no un evento concreto del CSV. Si obtienes el PDF con la definición del campo booleano, el retrieval ha funcionado: buscamos **contenido relevante**, no un archivo concreto.

Compara con la pregunta anterior de «cine gratuito»: ¿recuperan tipos de documento diferentes?

In [5]:
# Segunda pregunta de prueba (debería acercarse más a la FAQ)
pregunta2 = "¿Qué significa el campo GRATUITO?"
vector2 = embeddear_consulta(client, pregunta2)

res2 = collection.query(
    query_embeddings=[vector2],
    n_results=TOP_K,
    include=["documents", "metadatas", "distances"],
)

# Mejor resultado = posición 0 (el más similar)
mejor_fuente = Path(res2["metadatas"][0][0].get("source", "?")).name

print("Pregunta:", pregunta2)
print("Mejor fuente:", mejor_fuente)
print(formatear_contexto(res2)[:600], "...")

Pregunta: ¿Qué significa el campo GRATUITO?
Mejor fuente: 206974-3-agenda-eventos-culturales-100.pdf
--- Fragmento 1 (dist=0.4111) ---
Fuente: 206974-3-agenda-eventos-culturales-100.pdf
1. ID-EVENTO: es un campo clave, dentro de la web municipal, para uso interno.
2. TITULO: denominación del evento al que se refiere.
3. PRECIO: coste del evento/actividad. En muchos casos se trata de actividades gratuitas, como por
ejemplo todas las realizadas y organizadas en Bibliotecas Municipales, por lo que el campo
aparecerá vacío en muchas ocasiones.
4. GRATUITO: campo booleano, en el que figurará un “0” cuando no se ha marcado o un “1” cuando
se marque el evento como gratuito.
5. LARGA-DURACION: campo  ...


## Cierre y siguientes pasos

Has implementado el **retriever** manualmente:

```text
pregunta → embed → query en Chroma → top-K → contexto
```

Esto es la capa de **recuperación** del RAG. Sin un retriever fiable, el LLM no tiene buena información donde apoyarse.

El **siguiente paso** sería probar varias preguntas y valores de K para **evaluar** si el retrieval es bueno.